# MIST reaction/candidate conditioning: data, architecture, and results

**Purpose of this notebook**: a complete, source-linked record of (i) what
data was added beyond vanilla MIST and where it comes from, (ii) the
current architecture for using that data, (iii) every experiment/ablation
run and what each one specifically tests, and (iv) the resulting Tanimoto
similarity / cosine similarity / loss numbers, with the reasoning that
connects them to a recommendation. Written to support a slide deck
focused on: **what improvement comes from adding data (reactions/
candidates) on top of vanilla MIST, and what's still open.**

All numbers below are taken directly from completed Slurm job logs
(`results/*/split_1/output.log`, `logs/*.out`) and dedicated analysis
scripts under `analysis/` -- nothing is estimated. Where an analysis is
still running (Section 9, hyperparameter sweep), that's stated explicitly
with interim numbers, not final ones.

## 0. Scope and baseline setup

Every experiment in this notebook trains on the **`[M+H]+`-only subset of
NIST23** (`data/nist23/labels_mh_only.tsv`, 80,372 of 176,851 total
NIST23 spectra -- 45.5% of the full dataset). This was a deliberate choice
to develop and validate the reaction-conditioning work on a single,
clean adduct (matching the original MIST paper's own training regime,
which used `[M+H]+` exclusively) before generalizing to NIST23's full
13-adduct mix -- adduct diversity is a separate, larger source of
difficulty (Section 8) that reaction-conditioning does not address.

**Data prerequisite -- the collision-energy (CE) pooling fix.** Before any
of this work was practical, a bug in how NIST23's multi-collision-energy
spectra were processed had to be fixed. NIST23 (and NIST20) store each
compound's spectrum as several separate collision-energy acquisitions.
The pipeline as inherited assigned a chemical formula to each peak
**independently per collision-energy block**, then naively concatenated
all blocks' peaks with no deduplication before truncating to the top-N
most intense peaks -- meaning the same physical fragment observed at
multiple collision energies competed multiple times for a spot in the
truncated peak list, crowding out genuinely distinct fragments. The fix
(`data/nist23/subformulae/subform_50_repaired/`, built via
`run_scripts/submit_build_nist23_subform_repaired.sh`) merges all
collision-energy blocks' raw peaks FIRST (dedup by rounded m/z, keep max
intensity across blocks), then assigns one formula per spectrum --
matching exactly how the original paper's own training data (canopus_
train/csi2022) was built. This fix alone cut per-epoch training time
~25-30x (from ~25 min/epoch to ~45-70 sec/epoch on an H100) by removing
large amounts of redundant, wasted data-loader work, and is a
prerequisite for every experiment in this notebook -- without it,
iterating on the reaction-conditioning architecture below would have
taken ~9 hours per training run instead of ~1 hour.

**Base architecture for every run below** (unless noted): `hidden_size`
256, `peak_attn_layers` 2, `refine_layers` 4, `pairwise_featurization`
on, `no_diffs` on, `--embed-instrument` on, `morgan4096` target
fingerprint, `--magma-loss-lambda 8`, Adam-family optimizer,
`learning_rate` 0.00077, `--patience 20` early stopping.

## 1. Data sources: what's added beyond vanilla MIST, and where it comes from

Vanilla MIST predicts a fingerprint from the spectrum alone. This work
adds an OPTIONAL second input: known chemical context about the
compound's synthetic history, from two independent sources.

### 1a. `starting_materials` -- real reaction precursors

Built by `src/mist/build_reaction_metadata.py`, joining NIST23 compounds
(by InChIKey) against three public reaction-SMILES databases:

| Source file | Rows | Reaction database |
|---|---|---|
| `data/nist23/reaction_metadata_uspto.tsv` | 64,633 | USPTO_FULL (US patent reactions) |
| `data/nist23/reaction_metadata_cas.tsv` | 57,012 | CAS |
| `data/nist23/reaction_metadata_pistachio.tsv` | 56,222 | Pistachio |
| **Total** | **177,867** | |

Each row records a `product_smiles` (matched to a NIST23 compound),
`starting_materials` (`;`-separated SMILES of the reactants that produced
it), a `reaction_id`, and provenance metadata. A NIST23 compound can match
zero, one, or many reactions across these three sources; ~48% of unique
NIST23 compounds have at least one match (measured directly, Section 2).

### 1b. `candidates` -- algorithmically-generated decoy structures

A column on the same `reaction_metadata_*.tsv` files, populated for a
subset of USPTO rows by an external tool (`smartreact`) that generates
plausible-but-not-necessarily-correct near-product structures (small
structural edits/pairwise combinations of the real starting materials) --
NOT the real reactants, and not guaranteed correct. Coverage is much
lower: ~7.5-8% of unique NIST23 compounds.

### 1c. How this data is used

`datasets.attach_reactions` joins a batch of (spectrum, molecule) pairs to
the reaction metadata by InChIKey and attaches ALL matched records to each
`Spectra` object (capped at `--max-reactions-per-compound`, default 10, to
bound the influence of a few promiscuous compounds -- e.g. common
salt-forming counter-ions that incidentally match thousands of reactions).
**Leakage guard**: for `candidates`, any decoy matching the target
compound's OWN InChIKey is excluded, structurally, regardless of what the
source data contains. At training time, one matched reaction per compound
is picked at random each epoch (turning multiplicity into stochastic
augmentation); at standard eval time, aux data defaults to absent unless
explicitly steered (Section 5-7 below all depend on this behavior, in
different ways).

Each source's SMILES list is turned into ONE fixed-width vector via
`aux_featurizers.SmilesSetFeaturizer`: mean-pooled `morgan4096`
fingerprints (same fingerprint family/width as the model's own prediction
target) across every parseable SMILES in the list. An empty/absent list
becomes a zero vector.

## 2. Does this data actually correlate with the answer?

Before building any architecture, we checked whether the aux fingerprint
(mean-pooled Morgan FP of a compound's `starting_materials`, or of its
`candidates`) has a real relationship to the TRUE product fingerprint, or
is closer to noise. Full analysis notebook:
`notebooks/reaction_aux_fp_correlation.ipynb`.

**Method**: for every NIST23 compound with real aux data, compute Tanimoto
similarity between (a) the aux fingerprint and the true product
fingerprint ("real pairing"), and (b) the aux fingerprint and a randomly
shuffled OTHER compound's true fingerprint ("random pairing" -- the
no-signal baseline). Measured across all of NIST23 (not restricted to
`[M+H]+`), replicating the exact production featurization.

In [1]:
import pandas as pd

correlation_summary = pd.DataFrame([
    {"aux_source": "starting_materials", "n_compounds_measured": 22943,
     "real_pairing_mean_tanimoto": 0.3067, "random_pairing_mean_tanimoto": 0.0739,
     "diff": 0.2328, "pct_of_nist23_compounds_covered": 48.4},
    {"aux_source": "candidates", "n_compounds_measured": 3719,
     "real_pairing_mean_tanimoto": 0.3406, "random_pairing_mean_tanimoto": 0.0913,
     "diff": 0.2493, "pct_of_nist23_compounds_covered": 7.8},
])
correlation_summary

,aux_source,n_compounds_measured,real_pairing_mean_tanimoto,random_pairing_mean_tanimoto,diff,pct_of_nist23_compounds_covered
0,starting_materials,22943,0.3067,0.0739,0.2328,48.4
1,candidates,3719,0.3406,0.0913,0.2493,7.8


**Conclusion: yes, real and substantial signal.** Both sources sit at
~4x the random-pairing Tanimoto baseline (real pairing std was ~0.14-0.18
in the full analysis -- the effect is well outside noise). `candidates`
correlates slightly more strongly per-compound (plausible: it's
specifically constructed as a small structural edit near a plausible
product, whereas `starting_materials` can differ from the product by a
full synthetic transformation) but covers far fewer compounds. This
coverage-vs-strength tradeoff reappears directly in the results below.
This measurement is what justified building the architecture in Section
3 at all, rather than assuming the signal exists.

## 3. Architecture: two mechanisms tried, and why the second replaced the first

### 3a. v1 -- concatenation (`--aux-dim`, pre-existing before this session)

```
aux_fp   = mean_pool(Morgan4096(starting_materials_or_candidates))   # 4096-dim
aux_proj = LayerNorm(Linear(aux_fp))                                  # learned, 4096 -> aux_dim (e.g. 32)
head_in  = concat([pooled_spectrum_repr, aux_proj])                   # e.g. 256 + 32 = 288-dim
fp_pred  = sigmoid(fingerprint_head(head_in))
```

**Problem**: `aux_proj` compresses a 4096-dim fingerprint into a narrow
`aux_dim`-width vector via a generic learned linear map with no built-in
notion that it lives in the same space as the 4096-dim target. The model
has to rediscover, purely through indirect gradient signal on a SHARED
head (that also serves the ~50%+ of examples with no aux data at all),
that this compressed slice correlates bit-for-bit with the output --
despite Section 2 showing that signal is already there, undistorted, in
the RAW aux fingerprint.

### 3b. v2 -- gated residual blend (`--aux-gate`, built this session)

```
base_pred   = sigmoid(fingerprint_head(pooled_spectrum_repr))    # UNCHANGED spectrum-only path, always computed
base_score  = gate_head_base(pooled_spectrum_repr)                # per-bit score, 4096-dim
score_i     = gate_head_i([pooled_spectrum_repr, presence_flag_i])  # per source i (starting_materials, candidates)
              masked to -inf per-bit if source i is absent for this example
weights     = softmax([base_score, score_1, score_2, ...], dim=sources)   # per-bit, always sums to 1
fp_pred     = weight_base * base_pred + sum_i(weight_i * aux_fp_i)
```

Instead of compressing the aux fingerprint through a lossy projection,
its raw values participate DIRECTLY in the final blend, weighted by a
learned, per-bit trust score that is itself a function of the spectrum
(so the model can learn e.g. "this spectrum's fragmentation pattern
suggests the reaction barely changed the structure -> trust aux_fp
heavily" vs. "this looks like a big transformation -> trust the
spectrum-only prediction instead").

**Verified architectural guarantee** (checked with real forward passes,
not just by construction): when a source is absent, its pre-softmax
score is masked to `-inf`, so its post-softmax weight is EXACTLY 0 --
an absent-aux example's prediction is bit-for-bit identical to plain
MIST's output, and the model can never depend on aux data being present.
This was the explicit design requirement: the spectrum is always the
primary, always-available signal; reaction/candidate context is optional
additional evidence layered on top, never a dependency.

New CLI surface: `--aux-gate` (mutually exclusive with `--aux-dim`),
`--aux-sources starting_materials candidates` (restrict which sources are
built and used -- enables the isolation ablations in Section 6).

## 4. Experiments run

All on `[M+H]+`-only, repaired-subformula data, ~1hr wall-clock each
(post-CE-fix; the identical configuration took ~9hr/run pre-fix).

| Model | Mechanism | Sources | `--aux-dropout` |
|---|---|---|---|
| `nist23_fp_mist_mhplus_only` | none (vanilla MIST) | -- | n/a |
| `nist23_fp_mist_mhplus_aux_gate` | v2 gate | starting_materials + candidates | 0.2 |
| `nist23_fp_mist_mhplus_aux_gate_sm_only` | v2 gate | starting_materials only | 0.2 |
| `nist23_fp_mist_mhplus_aux_gate_cand_only` | v2 gate | candidates only | 0.2 |
| `nist23_fp_mist_mhplus_aux_gate_no_dropout` | v2 gate | starting_materials + candidates | 0.0 |

`--aux-dropout` randomly zeroes present aux data during training (so the
model doesn't learn to depend on it always being there); the `no_dropout`
variant tests whether this regularization is even necessary given the
gate's presence-flag already tells it explicitly when data is real.

For context, full-13-adduct-mix runs from earlier this session (using the
older v1 concatenation mechanism, before --aux-gate was built) are
referenced in Section 8.

## 5. Train / val / test convergence -- is the model over- or under-fit?

`train_fp_loss_epoch` is cosine loss (`1 - cosine_similarity`) on the
final training epoch; `cosine_sim = 1 - loss`. All values read directly
from each run's final logged epoch.

In [2]:
convergence = pd.DataFrame([
    {"model": "mhplus_only (vanilla)", "final_epoch": 92, "train_cosine_sim": 1 - 0.2148,
     "val_loss_final": 0.3347, "test_loss": 0.3267, "test_cosine_sim": 1 - 0.3267},
    {"model": "aux_gate (both sources)", "final_epoch": 49, "train_cosine_sim": 1 - 0.2165,
     "val_loss_final": 0.3412, "test_loss": 0.3362, "test_cosine_sim": 1 - 0.3362},
    {"model": "aux_gate_sm_only", "final_epoch": 66, "train_cosine_sim": 1 - 0.2045,
     "val_loss_final": 0.3415, "test_loss": 0.3367, "test_cosine_sim": 1 - 0.3367},
    {"model": "aux_gate_cand_only", "final_epoch": 69, "train_cosine_sim": 1 - 0.2282,
     "val_loss_final": 0.3341, "test_loss": 0.3276, "test_cosine_sim": 1 - 0.3276},
    {"model": "aux_gate_no_dropout", "final_epoch": 62, "train_cosine_sim": 1 - 0.1933,
     "val_loss_final": 0.3425, "test_loss": 0.3384, "test_cosine_sim": 1 - 0.3384},
])
convergence["train_val_cosine_gap"] = convergence["train_cosine_sim"] - (1 - convergence["val_loss_final"])
convergence

,model,final_epoch,train_cosine_sim,val_loss_final,test_loss,test_cosine_sim,train_val_cosine_gap
0,mhplus_only (vanilla),92,0.7852,0.3347,0.3267,0.6733,0.1199
1,aux_gate (both sources),49,0.7835,0.3412,0.3362,0.6638,0.1247
2,aux_gate_sm_only,66,0.7955,0.3415,0.3367,0.6633,0.1370
3,aux_gate_cand_only,69,0.7718,0.3341,0.3276,0.6724,0.1059
4,aux_gate_no_dropout,62,0.8067,0.3425,0.3384,0.6616,0.1492


**Diagnosis: overfitting, not underfitting.** Every model shows a
substantial, sustained train/val cosine-similarity gap (~0.11-0.14) that
had already plateaued -- both curves flat for 10+ epochs before
patience-20 triggered early stopping. This is the actual ceiling for the
current hyperparameters/regularization, not a "train longer" problem.
Directly motivates the hyperparameter sweep in Section 9.

## 6. Three progressively more honest ways to evaluate the gate

This is the core methodological finding of this investigation: **how you
evaluate a conditional-input model matters enormously**, and the naive
approach (Section 6a) gives an actively misleading answer.

**Threshold-calibration note applying to all Tanimoto numbers below**: the
default binarization threshold (0.5) is a poor fit for these sparse
(~0.87% bit density) fingerprints, and the `--aux-gate` softmax blend
pushes output magnitudes systematically lower as training improves --
raw `test_tanimoto` from training logs uses this default and is NOT
reliable for comparing across models. All numbers below are re-scored
with a swept threshold per model (`analysis/eval_checkpoint_threshold_
sweep.py`), reporting each model's OWN best threshold.

### 6a. Cold-start only (aux always absent) -- the naive comparison

In [3]:
coldstart = pd.DataFrame([
    {"model": "mhplus_only (vanilla)", "test_cosine": 0.6733, "best_thresh": 0.25, "test_tanimoto_best": 0.4661},
    {"model": "aux_gate (both)", "test_cosine": 0.6638, "best_thresh": 0.15, "test_tanimoto_best": 0.4431},
    {"model": "aux_gate_sm_only", "test_cosine": 0.6633, "best_thresh": 0.20, "test_tanimoto_best": 0.4474},
    {"model": "aux_gate_cand_only", "test_cosine": 0.6724, "best_thresh": 0.20, "test_tanimoto_best": 0.4599},
    {"model": "aux_gate_no_dropout", "test_cosine": 0.6616, "best_thresh": 0.20, "test_tanimoto_best": 0.4455},
])
coldstart

,model,test_cosine,best_thresh,test_tanimoto_best
0,mhplus_only (vanilla),0.6733,0.25,0.4661
1,aux_gate (both),0.6638,0.15,0.4431
2,aux_gate_sm_only,0.6633,0.20,0.4474
3,aux_gate_cand_only,0.6724,0.20,0.4599
4,aux_gate_no_dropout,0.6616,0.20,0.4455


In this condition, EVERY gate variant is flat or very slightly WORSE than
vanilla MIST (0.443-0.460 vs. 0.466). Taken alone, this says "the aux
data doesn't help." It's an incomplete picture: this condition never lets
the gate use real aux data at all.

### 6b. Reaction-sensitivity: does the gate use real data when it has it?

For every test compound with a real match, `src/mist/analyze_reaction_
sensitivity.py` runs inference BOTH with that real match steered in AND
with no reaction (same compound, same spectrum, same true fingerprint) --
isolating whether supplying real aux data changes and improves the
prediction. (Uses each model's default 0.5 threshold internally; absolute
values here aren't comparable to 6a/6c, only the within-model with-vs-
without comparison is meaningful.)

In [4]:
sensitivity = pd.DataFrame([
    {"model": "aux_gate (both)", "source": "starting_materials", "n": 4061,
     "no_reaction_tanimoto": 0.1911, "with_reaction_tanimoto": 0.2826,
     "diff": 0.0915, "pct_compounds_improved": 68.9, "within_compound_std": 0.0676},
    {"model": "aux_gate (both)", "source": "candidates", "n": 614,
     "no_reaction_tanimoto": 0.2183, "with_reaction_tanimoto": 0.3422,
     "diff": 0.1240, "pct_compounds_improved": 77.4, "within_compound_std": 0.0093},
    {"model": "aux_gate_sm_only", "source": "starting_materials", "n": 4061,
     "no_reaction_tanimoto": 0.2761, "with_reaction_tanimoto": 0.3462,
     "diff": 0.0702, "pct_compounds_improved": 65.1, "within_compound_std": 0.0595},
    {"model": "aux_gate_cand_only", "source": "candidates", "n": 609,
     "no_reaction_tanimoto": 0.4032, "with_reaction_tanimoto": 0.4735,
     "diff": 0.0703, "pct_compounds_improved": 64.0, "within_compound_std": 0.0074},
    {"model": "aux_gate_no_dropout", "source": "starting_materials", "n": 4061,
     "no_reaction_tanimoto": 0.3313, "with_reaction_tanimoto": 0.3947,
     "diff": 0.0634, "pct_compounds_improved": 64.4, "within_compound_std": 0.0629},
    {"model": "aux_gate_no_dropout", "source": "candidates", "n": 612,
     "no_reaction_tanimoto": 0.3454, "with_reaction_tanimoto": 0.4548,
     "diff": 0.1094, "pct_compounds_improved": 69.6, "within_compound_std": 0.0086},
])
sensitivity

,model,source,n,no_reaction_tanimoto,with_reaction_tanimoto,diff,pct_compounds_improved,within_compound_std
0,aux_gate (both),starting_materials,4061,0.1911,0.2826,0.0915,68.9,0.0676
1,aux_gate (both),candidates,614,0.2183,0.3422,0.1240,77.4,0.0093
2,aux_gate_sm_only,starting_materials,4061,0.2761,0.3462,0.0702,65.1,0.0595
3,aux_gate_cand_only,candidates,609,0.4032,0.4735,0.0703,64.0,0.0074
4,aux_gate_no_dropout,starting_materials,4061,0.3313,0.3947,0.0634,64.4,0.0629
5,aux_gate_no_dropout,candidates,612,0.3454,0.4548,0.1094,69.6,0.0086


**Every variant, every source: real, positive, substantial improvement**
when a match is actually supplied (+0.06 to +0.12 Tanimoto, 64-77% of
matched compounds improve). Small within-compound spread (std 0.007-
0.068) relative to these gaps indicates guidance-like behavior (a useful
soft prior) rather than memorization (which would show large, erratic
spread across different matched reactions for the same compound).

### 6c. The real answer: population-weighted, realistic mixed-aux eval

`analysis/eval_realistic_mixed_aux.py` gives EVERY test compound its
ACTUAL real condition (real starting_materials if matched, real
candidates if matched, nothing if neither) in one pass over the full test
set (n=8,179; 49.7% have >=1 real match, 50.3% have none) -- the honest,
deployable number.

In [5]:
realistic = pd.DataFrame([
    {"model": "mhplus_only (vanilla)", "whole_set_cosine": 0.6733, "has_either_cosine": 0.6844, "neither_cosine": 0.6623,
     "best_thresh": 0.25, "whole_set_tanimoto_best": 0.4661},
    {"model": "aux_gate (both)", "whole_set_cosine": 0.7114, "has_either_cosine": 0.7673, "neither_cosine": 0.6563,
     "best_thresh": 0.15, "whole_set_tanimoto_best": 0.5095},
    {"model": "aux_gate_sm_only", "whole_set_cosine": 0.7103, "has_either_cosine": 0.7645, "neither_cosine": 0.6569,
     "best_thresh": 0.15, "whole_set_tanimoto_best": 0.5090},
    {"model": "aux_gate_cand_only", "whole_set_cosine": 0.6762, "has_either_cosine": 0.6905, "neither_cosine": 0.6621,
     "best_thresh": 0.20, "whole_set_tanimoto_best": 0.4650},
    {"model": "aux_gate_no_dropout", "whole_set_cosine": 0.7093, "has_either_cosine": 0.7644, "neither_cosine": 0.6550,
     "best_thresh": 0.20, "whole_set_tanimoto_best": 0.5125},
])
realistic["tanimoto_gain_vs_vanilla"] = realistic["whole_set_tanimoto_best"] - 0.4661
realistic["cosine_gain_vs_vanilla"] = realistic["whole_set_cosine"] - 0.6733
realistic

,model,whole_set_cosine,has_either_cosine,neither_cosine,best_thresh,whole_set_tanimoto_best,tanimoto_gain_vs_vanilla,cosine_gain_vs_vanilla
0,mhplus_only (vanilla),0.6733,0.6844,0.6623,0.25,0.4661,0.0000,0.0000
1,aux_gate (both),0.7114,0.7673,0.6563,0.15,0.5095,0.0434,0.0381
2,aux_gate_sm_only,0.7103,0.7645,0.6569,0.15,0.5090,0.0429,0.0370
3,aux_gate_cand_only,0.6762,0.6905,0.6621,0.20,0.4650,-0.0011,0.0029
4,aux_gate_no_dropout,0.7093,0.7644,0.6550,0.20,0.5125,0.0464,0.0360


**This reverses the 6a conclusion. On the realistic full test set:**

- **`aux_gate_no_dropout` and `aux_gate` (both sources) clearly beat
  vanilla MIST**: 0.5125 and 0.5095 Tanimoto vs. vanilla's 0.4661 -- a
  **+0.043 to +0.046 Tanimoto gain** (+9.2% to +9.9% relative), and
  +0.036 to +0.038 cosine similarity gain.
- `aux_gate_sm_only` is essentially tied with the full both-sources model
  (0.5090 vs. 0.5095) -- most of the total benefit traces to
  `starting_materials` (49.7% coverage), not `candidates` (7.5%
  coverage).
- `aux_gate_cand_only` alone is roughly flat vs. vanilla (0.4650 vs.
  0.4661) -- its per-compound benefit is real (6b) but coverage is too
  low for it to move the whole-set number on its own.

**Why 6a and 6c disagree**: 6a forces every compound into the cold-start
condition, measuring only the gate's small cold-start cost. 6c gives each
compound its real condition, and the with-aux upside (6b: +0.06 to +0.12
on ~50% of compounds) more than pays for that cost on the other ~50%.

**Best current configuration: `aux_gate_no_dropout`** -- marginally ahead
of default-dropout `aux_gate` (0.5125 vs. 0.5095), suggesting
`--aux-dropout` regularization may be unnecessary given the gate's
explicit presence-flag mechanism.

## 7. Bottom line: value of adding reaction/candidate data

| | Vanilla MIST | + reaction/candidate data (`aux_gate_no_dropout`) | Gain |
|---|---|---|---|
| Whole-test-set Tanimoto | 0.4661 | **0.5125** | **+0.0464 (+10.0% relative)** |
| Whole-test-set cosine similarity | 0.6733 | **0.7093** | **+0.0360** |
| On the ~50% of compounds WITH a match (thresh=0.20) | 0.4661* | **0.5877** | **+0.1216** |
| On the ~50% of compounds WITHOUT a match (thresh=0.20) | 0.4661* | 0.4384 | -0.0277 (small cost, more than offset) |

\* vanilla MIST has no concept of "with/without a match" -- shown as the
same flat number for reference.

**This is the headline number for the slide deck**: adding known reaction
context, used via the gated architecture, delivers a genuine +10% relative
improvement in whole-test-set Tanimoto similarity over vanilla MIST, at
zero risk to the ~50% of compounds that have no such data available
(architecturally guaranteed identical-or-better than vanilla in that
case, per Section 3b's verified absent-source behavior).

## 8. Adduct diversity: a separate, larger lever (not addressed by this work)

For context: the same architecture trained on NIST23's full 13-adduct mix
(not just `[M+H]+`) shows a substantially larger gap purely from adduct
heterogeneity -- using the OLDER v1 concatenation mechanism (before
`--aux-gate` existed), raw default-threshold numbers, not directly
comparable in absolute terms to Sections 5-7 but directionally clear:

In [6]:
adduct_comparison = pd.DataFrame([
    {"config": "full 13-adduct mix, no aux data", "test_tanimoto_raw_default_thresh": 0.3232},
    {"config": "full 13-adduct mix, --embed-adduct (explicit adduct feature)", "test_tanimoto_raw_default_thresh": 0.3470},
    {"config": "full 13-adduct mix, v1 concat aux (starting_materials+candidates)", "test_tanimoto_raw_default_thresh": 0.3690},
    {"config": "[M+H]+ only, no aux (this notebook's vanilla baseline)", "test_tanimoto_raw_default_thresh": 0.4211},
])
adduct_comparison

,config,test_tanimoto_raw_default_thresh
0,"full 13-adduct mix, no aux data",0.3232
1,"full 13-adduct mix, --embed-adduct (explicit a...",0.3470
2,"full 13-adduct mix, v1 concat aux (starting_ma...",0.3690
3,"[M+H]+ only, no aux (this notebook's vanilla b...",0.4211


Restricting to `[M+H]+` alone was, by itself, the single largest lever
identified this session -- larger than any aux-conditioning variant.
NIST23's 13-adduct mix (vs. the original paper's `[M+H]+`-only training
regime) is a real, separate source of difficulty that the gate
architecture does not address and was not designed to address.
**Open next step**: repeat Sections 4-7's ablations on the full adduct
mix once `--aux-gate` is adopted as the standard mechanism, to see if the
reaction-conditioning gain holds up outside the clean `[M+H]+` subset.

## 9. Hyperparameter sweep -- interim result (IN PROGRESS, not final)

Motivated directly by Section 5's overfitting diagnosis. Reruns the
earlier (pre-CE-fix, stale) sweep on repaired, `[M+H]+`-only, full-size
data with no subsampling (the old sweep subsampled to 10%/15% specifically
because the pre-fix pipeline was too slow to search full data; post-fix,
full epochs run in ~1 min). Search space widened: `weight_decay` now spans
`[1e-7, 1e-6, 1e-5, 1e-4, 1e-3]` (previously topped out at a negligible
`1e-6`), and `batch_size` (`[64, 128, 256]`) is newly included -- neither
had been meaningfully explored before. Running as 3 concurrent trials on
3 real GPUs (`--gres=gpu:l40s:3 --max-concurrent 3`), 30 trials planned.

In [7]:
# Interim: 9/30 trials completed as of this notebook's last run. NOT final.
hyperopt_interim = pd.DataFrame([
    {"trial": 0, "val_loss": 0.3301, "weight_decay": 1e-7, "batch_size": 128, "spectra_dropout": 0.1, "note": "baseline config (same as all Section 4-7 runs)"},
    {"trial": 6, "val_loss": 0.3517, "weight_decay": 1e-7, "batch_size": 256, "spectra_dropout": 0.4, "note": ""},
    {"trial": 1, "val_loss": 0.3562, "weight_decay": 1e-5, "batch_size": 256, "spectra_dropout": 0.2, "note": ""},
    {"trial": 3, "val_loss": 0.3735, "weight_decay": 1e-6, "batch_size": 128, "spectra_dropout": 0.3, "note": ""},
    {"trial": 2, "val_loss": 0.4077, "weight_decay": 1e-7, "batch_size": 256, "spectra_dropout": 0.3, "note": ""},
])
hyperopt_interim

,trial,val_loss,weight_decay,batch_size,spectra_dropout,note
0,0,0.3301,1.000000e-07,128,0.1,baseline config (same as all Section 4-7 runs)
1,6,0.3517,1.000000e-07,256,0.4,
2,1,0.3562,1.000000e-05,256,0.2,
3,3,0.3735,1.000000e-06,128,0.3,
4,2,0.4077,1.000000e-07,256,0.3,


**Interim finding (9/30 trials complete, DO NOT treat as final)**: the
baseline configuration already used in every Section 4-7 run (trial 0,
`weight_decay=1e-7`, `batch_size=128`, `spectra_dropout=0.1`) is still the
best of all completed trials. None of the widened `weight_decay` values or
new `batch_size` options tested so far have beaten it. This does not mean
the overfitting gap (Section 5) is unfixable -- 21 trials remain,
including combinations not yet tried. Re-run this cell against
`results/nist23_hyperopt_mist_mhplus_repaired/best_trial.yaml` once the
sweep completes for the final answer.

## 10. Summary and open points of improvement

**Confirmed, with evidence:**
1. The CE-pooling fix was a genuine, necessary correctness fix (not
   optional), and delivered a ~25-30x training speedup as a side effect.
2. The reaction/candidate aux signal is real (~4x random-baseline
   Tanimoto correlation with the true product, Section 2).
3. The gated architecture (v2) genuinely exploits that signal on matched
   compounds (+0.06 to +0.12 Tanimoto, Section 6b) with guidance-like,
   not memorization-like, behavior.
4. On the realistic, full test population, gating delivers a **net +10%
   relative Tanimoto improvement over vanilla MIST** (Section 7) at no
   architectural risk to compounds lacking aux data.
5. A cold-start-only comparison (Section 6a) would have hidden this
   entirely and given the wrong answer -- a durable methodological
   lesson for evaluating any conditionally-available-data feature.

**Open points of improvement, ranked by expected leverage:**
1. **Adduct diversity** (Section 8) -- the single largest lever measured
   this session, unaddressed by any work in this notebook. Repeat
   Sections 4-7 on the full 13-adduct mix once `--aux-gate` is adopted.
2. **Overfitting** (Section 5) -- real, ~0.13 train/val cosine gap on
   every model. Hyperparameter sweep in progress (Section 9); interim
   result suggests the fix isn't simply "more weight decay" or "different
   batch size" alone -- may need architecture-level regularization or
   more/better training data (see point 4).
3. **`candidates` coverage** (Sections 2, 6) -- the stronger of the two
   signals per-compound, but only 7.5-8% coverage. Extending
   `candidates` generation to more compounds is a data-engineering
   project, not a training experiment, and could plausibly close more
   gap than further architecture tuning on the current sources.
4. **Simulated/forward-augmented training data** -- the original MIST
   paper trains with up to 40% synthetic (spectrum, fingerprint) pairs
   from a forward simulation model; NIST23 currently has none. Not yet
   built for NIST23 (would require training a separate forward model);
   plausibly relevant to point 2 given the paper's own ablations show it
   helps generalization, though the effect size there was modest on their
   dataset.
5. **Residual/"warm-start" refinement of the gate** -- a variant where
   the aux fingerprint is treated as a reference point and the model
   predicts a correction/delta rather than an independent blend weight.
   Not yet built; only worth pursuing if it can be shown the gate is
   leaving headroom on the table (e.g. by checking how close
   `with_reaction_tanimoto`, Section 6b, already is to simply copying
   `aux_fp` forward unchanged) -- not yet measured.

## 11. All reaction-metadata sources + full adduct mix

Two new runs, submitted and completed 2026-08-13, motivated by two open items from
Sections 8/10: (a) a new reaction-metadata source
(`reaction_metadata_suong.tsv`, CAS name-reaction extraction, 55,360 rows)
added to the existing uspto/cas/pistachio lineup, and (b) finally running
the `--aux-gate` architecture on the full 13-adduct mix instead of only
`[M+H]+` (Section 8's full-mix numbers used the older `--aux-dim` concat
mechanism, not the gate).

| Model | Adduct scope | Reaction sources | Job ID |
|---|---|---|---|
| `nist23_fp_mist_mhplus_aux_gate_all` | `[M+H]+`-only | uspto+cas+pistachio+suong | 20343414 |
| `nist23_fp_mist_embed_adduct_aux_gate_all` | full 13-adduct mix + `--embed-adduct` | uspto+cas+pistachio+suong | 20343416 |

No code changes required -- `--reaction-metadata-file` already accepts
multiple files (`datasets.attach_reactions` concatenates matches across
all of them), so adding `suong` is purely a data change. Full config:
`run_scripts/submit_nist23_fp_mist_mhplus_aux_gate_all.sh`,
`run_scripts/submit_nist23_fp_mist_embed_adduct_aux_gate_all.sh`. See
`docs/nist23_ablation_plan.md` arm (iv) for the tracked ablation entry.

Eval (threshold sweep, Section 6a-style; realistic mixed-aux, Section
6c-style) is chained via `sbatch --dependency=afterok` on both training
jobs (`run_scripts/submit_all_metadata_eval.sh`, job 20343567) and will
populate the table below once both training runs finish.


In [ ]:
# Results from results/all_metadata_eval_output.log (job 20343567,
# threshold sweep + realistic mixed-aux eval), 2026-08-13.
all_metadata_results = pd.DataFrame([
    {"model": "mhplus_aux_gate_all (4 sources)", "adduct_scope": "[M+H]+ only",
     "coldstart_test_cosine": 0.6650, "coldstart_test_tanimoto_best": 0.4463, "best_thresh": 0.2,
     "realistic_whole_set_cosine": 0.7115, "realistic_has_either_cosine": 0.7663,
     "realistic_neither_cosine": 0.6574, "realistic_whole_set_tanimoto_best": 0.5105,
     "pct_has_either": 49.7},
    {"model": "embed_adduct_aux_gate_all (4 sources)", "adduct_scope": "full 13-adduct mix",
     "coldstart_test_cosine": 0.6508, "coldstart_test_tanimoto_best": 0.4250, "best_thresh": 0.2,
     "realistic_whole_set_cosine": 0.6975, "realistic_has_either_cosine": 0.7597,
     "realistic_neither_cosine": 0.6474, "realistic_whole_set_tanimoto_best": 0.4883,
     "pct_has_either": 44.6},
])
all_metadata_results


**Findings:**

1. **Adding `suong` (4th source) to `[M+H]+`-only aux-gate roughly matches
   the existing 3-source `aux_gate` run** (Section 7: 0.5095/0.7093;
   here: 0.5105/0.7115 realistic-mixed-aux tanimoto/cosine) -- essentially
   flat, not a further gain. Coverage (`has_either`, 49.7%) is unchanged
   from the 3-source run too, so `suong`'s CAS name-reaction matches
   mostly overlap compounds already matched by uspto/cas/pistachio rather
   than reaching new ones.
2. **First full-adduct-mix run on the `--aux-gate` architecture
   (`embed_adduct_aux_gate_all`)**: realistic-mixed-aux whole-set tanimoto
   0.4883, cosine 0.6975 -- both below the `[M+H]+`-only aux-gate number
   (0.5105/0.7115), consistent with Section 8's earlier finding that
   adduct diversity is a real, separate cost the gate architecture
   doesn't erase. Coverage is slightly lower too (44.6% vs 49.7%
   `has_either`) since the full-mix test set is larger and less
   reaction-matched on average.
3. Cold-start-only numbers (thresh=0.2 test tanimoto: 0.4463 for
   mhplus, 0.4250 for embed_adduct) again undersell both models relative
   to the realistic mixed-aux eval, reproducing Section 6a/6c's
   methodological point.

**Bottom line**: the new `suong` source doesn't move the needle on
`[M+H]+`-only aux-gate performance -- diminishing returns from adding
more reaction sources once uspto+cas+pistachio already cover the
matchable population. Running aux-gate on the full adduct mix confirms
Section 8's adduct-diversity finding holds even with reaction
conditioning: full-mix numbers stay below `[M+H]+`-only across the
board.
